source API URL : "https://geocoding-api.open-meteo.com/v1/search?name=kovilpatti&count=10&language=en&format=json"

JSON Target File Path : "abfss://working-labs@datalakestorageaccountname.dfs.core.windows.net/bronze/geo-location/
"

In [0]:
#geoLocationSourceAPIURL = "https://geocoding-api.open-meteo.com/v1/search?name=kovilpatti&count=10&language=en&format=json"

geoLocationSourceAPIBaseURL = "https://geocoding-api.open-meteo.com/v1/search?name="
geoLocationSourceAPIOptions = "&count=10&language=en&format=json"


geoLocationSinkLayerName = "bronze"
geoLocationSinkStorageAccountName = "adlsadatalakehouse"
geoLocationSinkFolderName = "geo-location"
geoLocationSinkFolderPath = f"abfss://{geoLocationSinkLayerName}@{geoLocationSinkStorageAccountName}.dfs.core.windows.net/{geoLocationSinkFolderName}"

In [0]:
import requests
import json
import pandas as pds

In [0]:
dailyPricingMarketNameDF = spark.sql("select MARKET_NAME from pricing_analytics.gold.reporting_dim_market_gold")

In [0]:
marketNames = [dailypricingMarketNames["MARKET_NAME"] for dailypricingMarketNames in dailyPricingMarketNameDF.collect()]
geoLocationAPIResponseList = []
for marketName in marketNames:
    geoLocationSourceAPIURL = f"{geoLocationSourceAPIBaseURL}{marketName}{geoLocationSourceAPIOptions}"
    geoLocationAPIResponse = requests.get(geoLocationSourceAPIURL).json()
    if isinstance(geoLocationAPIResponse,dict):
        geoLocationAPIResponseList.append(geoLocationAPIResponse)
geoLocationSparkRDD = sc.parallelize(geoLocationAPIResponseList)
geoLocationSparkDF = spark.read.json(geoLocationSparkRDD)
(geoLocationSparkDF
    .filter("results.admin1 is not null")
    .write
    .mode("overwrite")
    .json(geoLocationSinkFolderPath))

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
geoLocationBronzeDF = (spark
                       .read
                       .json(geoLocationSinkFolderPath))
display(geoLocationBronzeDF)